# City Pulse — Urban Liveability Analysis
### A complete data science project walkthrough

**Topics covered:** Data loading · EDA · Preprocessing · K-Means clustering · Linear / Ridge / Lasso regression · Model evaluation · Interpretation

---

## 0. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from src.data.loader import (
    load_raw, validate, clean, add_features, split, NUMERIC_FEATURES, TARGET, FEATURE_DISPLAY_NAMES
)
from src.visualization.eda_plots import (
    overview, plot_distributions, plot_correlation_matrix,
    plot_scatter, plot_boxplots, continent_summary, plot_continent_bars,
    detect_outliers_iqr, detect_outliers_zscore,
)
from src.models.clustering import (
    run_kmeans, plot_pca_clusters, plot_radar, plot_elbow,
    describe_clusters, plot_silhouette, FEATURE_GROUPS,
)
from src.models.regression import (
    train_model, train_all, evaluate, evaluate_all, cross_validate,
    get_coefficients, plot_predicted_vs_actual, plot_residuals,
    plot_learning_curves, plot_regularisation_path, MODELS,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Ready!')

---
## 1. Data Loading & Validation

In [ ]:
# Load and inspect
df_raw = load_raw()
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
# Data types and nulls
df_raw.info()

In [ ]:
# Validate — should print all-clear
validate(df_raw)

In [ ]:
# Clean and engineer features
df = clean(df_raw)
df = add_features(df)
print('Engineered columns:', [c for c in df.columns if c not in df_raw.columns])
df.head()

---
## 2. Exploratory Data Analysis

In [ ]:
# Statistical summary
overview(df)

In [ ]:
# Feature distributions
fig = plot_distributions(df)
plt.show()

In [ ]:
# Boxplots — spot outliers quickly
fig = plot_boxplots(df)
plt.show()

In [ ]:
# Correlation matrix
fig = plot_correlation_matrix(df)
plt.show()

In [ ]:
# Scatter: air quality vs liveability (try different pairs!)
fig = plot_scatter(df, 'air_quality', 'liveability_score')
plt.show()

In [ ]:
# Liveability by continent
print(continent_summary(df))
fig = plot_continent_bars(df)
plt.show()

In [ ]:
# Outlier detection
outliers_iqr = detect_outliers_iqr(df)
print('IQR outliers:')
print(outliers_iqr[outliers_iqr['any_outlier']][['city'] + NUMERIC_FEATURES])

outliers_z = detect_outliers_zscore(df)
print('\nZ-score outliers:')
print(outliers_z[outliers_z['any_outlier']][['city'] + NUMERIC_FEATURES])

> **Exercise:** Which cities are consistently flagged as outliers? What do they have in common? Does removing them change the correlation matrix significantly?

---
## 3. Data Preprocessing

In [ ]:
# Scale features (z-score)
scaler = StandardScaler()
X = df[NUMERIC_FEATURES]
y = df[TARGET]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# IMPORTANT: fit scaler ONLY on training data to avoid data leakage
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=NUMERIC_FEATURES, index=X_train_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test_raw),     columns=NUMERIC_FEATURES, index=X_test_raw.index)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print('Train mean (should be ~0):', X_train.mean().round(3).values)
print('Test mean (not guaranteed ~0):', X_test.mean().round(3).values)

> **Key concept:** We fit the scaler on training data only. If we scaled the whole dataset first, information from the test set would "leak" into the model — inflating performance.

---
## 4. K-Means Clustering

In [ ]:
# Elbow method + silhouette scores to choose K
X_all_scaled = scaler.fit_transform(df[NUMERIC_FEATURES])
fig = plot_elbow(X_all_scaled)
plt.show()

In [ ]:
# Run K-Means with chosen K
K = 3  # <-- change this based on the elbow chart
df_clustered, km_model, km_scaler, feats = run_kmeans(df, k=K, feature_group='all')

# Show cluster assignments
describe_clusters(df_clustered)

In [ ]:
# PCA scatter — 2D visualisation of clusters
fig = plot_pca_clusters(df_clustered)
plt.show()

In [ ]:
# Radar chart — feature profile per cluster
fig = plot_radar(df_clustered)
plt.show()

In [ ]:
# Silhouette plot
labels = df_clustered['cluster'].values
fig = plot_silhouette(X_all_scaled, labels, K)
plt.show()

In [ ]:
# Try environmental features only
df_env, _, _, _ = run_kmeans(df, k=3, feature_group='environmental')
fig = plot_pca_clusters(df_env, FEATURE_GROUPS['environmental'])
plt.title('Environmental features only')
plt.show()

> **Exercise:** Does changing the feature group produce meaningfully different clusters? Try K=4 and K=5. Write a 2-sentence interpretation for each cluster.

---
## 5. Regression — Predicting Liveability

In [ ]:
# Fit plain linear regression
lr = train_model(X_train, y_train, 'Linear Regression')
metrics = evaluate(lr, X_test, y_test, 'Linear Regression')
print(metrics)

In [ ]:
# Predicted vs actual
city_names_test = df.loc[X_test.index, 'city']
fig = plot_predicted_vs_actual(lr, X_test, y_test, city_names_test, 'Linear Regression')
plt.show()

In [ ]:
# Residual diagnostics
fig = plot_residuals(lr, X_test, y_test, 'Linear Regression')
plt.show()

In [ ]:
# Coefficients
get_coefficients(lr)

In [ ]:
# 5-fold cross-validation (uses Pipeline internally — no leakage)
cv_res = cross_validate(df, NUMERIC_FEATURES, 'Linear Regression')
print(cv_res)

---
## 6. Model Comparison

In [ ]:
# Train all models
fitted = train_all(X_train, y_train)

# Compare on test set
results = evaluate_all(fitted, X_test, y_test)
results

In [ ]:
# Regularisation path — Ridge
fig = plot_regularisation_path(X_train, y_train, model_type='ridge')
plt.show()

In [ ]:
# Regularisation path — Lasso (notice coefficients going to zero = feature selection)
fig = plot_regularisation_path(X_train, y_train, model_type='lasso')
plt.show()

In [ ]:
# Learning curves
fig = plot_learning_curves(df, model_name='Linear Regression')
plt.show()

> **Discussion:** Do the training and validation curves converge? What does a large gap indicate? What does it mean when both curves are low?

---
## 7. Interpretation & Conclusions

> **Your task:** Write a 500-word narrative below answering:
> 1. Which 2–3 features best predict liveability? Are you surprised?
> 2. Which cluster of cities would you describe as 'liveable but expensive'?
> 3. Does air quality dominate over social factors?
> 4. What are 2 limitations of this dataset and how could you address them?
> 5. If you were advising a mayor, which single feature would you tell them to prioritise and why?

*(Write your interpretation here)*

---
## 8. Extension Challenges

In [ ]:
# Challenge 1: Random Forest feature importances
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=NUMERIC_FEATURES)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([FEATURE_DISPLAY_NAMES[f] for f in importances.index], importances.values, color='#7F77DD')
ax.set_xlabel('Feature importance')
ax.set_title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

rf_metrics = evaluate(rf, X_test, y_test, 'Random Forest')
print(rf_metrics)

In [ ]:
# Challenge 2: SHAP explainability (requires: pip install shap)
try:
    import shap
    explainer = shap.LinearExplainer(lr, X_train)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, feature_names=[FEATURE_DISPLAY_NAMES[f] for f in NUMERIC_FEATURES])
except ImportError:
    print('Install shap: pip install shap')

In [ ]:
# Challenge 3: DBSCAN clustering
from sklearn.cluster import DBSCAN

dbs = DBSCAN(eps=1.2, min_samples=3)
db_labels = dbs.fit_predict(X_all_scaled)
df_db = df.copy()
df_db['cluster'] = db_labels
df_db['cluster_label'] = df_db['cluster'].apply(lambda x: 'Noise' if x == -1 else f'Cluster {x+1}')

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
print(f'DBSCAN: {n_clusters} clusters, {n_noise} noise points')
print(df_db[df_db['cluster'] == -1][['city', 'liveability_score']])